# Task 2.4
Before training, verify your analytical gradients from Task 2.3 against numerical finite differences. For a randomly initialised network, compute both the analytical gradient $\frac{\partial L}{\partial W_{ij}^{(1)}}$ and the finite difference approximation:

$$
\frac{\partial L}{\partial W_{ij}^{(1)}} \approx
\frac{
L\left(W_{ij}^{(1)}+\epsilon\right)
-L\left(W_{ij}^{(1)}-\epsilon\right)
}{
2\epsilon
}
\tag{8}
$$

using $\epsilon=10^{-5}$

In [1]:
# %%capture
# %run Task2.ipynb

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import math
import copy

## Equation
The two equations I have derived and will compare are:

**Gradient with respect to $W_n^{(1)}$:**

$$
\begin{aligned}
\frac{\partial L}{\partial W_n^{(1)}}
={}&
\frac{2}{N_d}\sum_{i=1}^{N_d}
\left(NN(t_i)-x(t_i)\right)
W_n^{(2)}t_i
\left(1-\tanh^2(z_{n,i})\right)
\\[6pt]
&+\lambda_{phys}\frac{2}{N_c}
\sum_{i=1}^{N_c}r_i
\Bigg[
-4W_n^{(2)}W_n^{(1)}
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
-2W_n^{(2)}(W_n^{(1)})^2t_i
\left(1-\tanh^2(z_{n,i})\right)^2
\\
&\quad
+4W_n^{(2)}(W_n^{(1)})^2t_i
\tanh^2(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{\mu}{m}
\Bigg(
W_n^{(2)}
\left(1-\tanh^2(z_{n,i})\right)
\\
&\qquad
-2W_n^{(2)}W_n^{(1)}t_i
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\Bigg)
\\
&\quad
+\frac{k}{m}W_n^{(2)}t_i
\left(1-\tanh^2(z_{n,i})\right)
\Bigg]
\\[6pt]
&+\lambda_{IC}
\Bigg[
2\left(
\sum_{j=1}^{H}W_j^{(2)}W_j^{(1)}
\left(1-\tanh^2(b_j^{(1)})\right)
\right)
W_n^{(2)}
\left(1-\tanh^2(b_n^{(1)})\right)
\Bigg]
\end{aligned}
$$

**Gradient with respect to $W_n^{(2)}$:**

$$
\begin{aligned}
\frac{\partial L}{\partial W_n^{(2)}}
={}&
\frac{2}{N_d}\sum_{i=1}^{N_d}
\left(NN(t_i)-x(t_i)\right)
\tanh(z_{n,i})
\\[6pt]
&+\lambda_{phys}\frac{2}{N_c}
\sum_{i=1}^{N_c}r_i
\Bigg[
-2(W_n^{(1)})^2
\tanh(z_{n,i})
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{\mu}{m}W_n^{(1)}
\left(1-\tanh^2(z_{n,i})\right)
\\
&\quad
+\frac{k}{m}\tanh(z_{n,i})
\Bigg]
\\[6pt]
&+\lambda_{IC}
\Bigg[
2\left(
\sum_{j=1}^{H}W_j^{(2)}
\tanh(b_j^{(1)})+b^{(2)}-1
\right)
\tanh(b_n^{(1)})
\\
&\quad
+2\left(
\sum_{j=1}^{H}W_j^{(2)}W_j^{(1)}
\left(1-\tanh^2(b_j^{(1)})\right)
\right)
W_n^{(1)}
\left(1-\tanh^2(b_n^{(1)})\right)
\Bigg]
\end{aligned}
$$

**Loss function:**
$$
L = L_{data} + \lambda_{phys}\cdot L_{phys} + \lambda_{IC}\cdot L_{IC}
$$
**Loss function expanded:**

$$
\begin{aligned}
L ={}&
\frac{1}{N_d}\sum_{i=1}^{N_d}
\left(
NN(t_i)-e^{-2.5t_i}
\left(2\cdot0.5\cdot\cos(-0.11+22.22t_i)\right)
\right)^2
\\[6pt]
&+\lambda_{phys}\frac{1}{N_c}\sum_{i=1}^{N_c}
\left(
\frac{d^2NN}{dt^2}(t_i)
+\frac{\mu}{m}\frac{dNN}{dt}(t_i)
+\frac{k}{m}NN(t_i)
\right)^2
\\[6pt]
&+\lambda_{IC}
\left[
\left(NN(0)-1\right)^2
+\left(\frac{dNN}{dt}(0)\right)^2
\right]
\end{aligned}
$$

**Neural Network**
$$
NN(t_i) = \sum_{n=1}^{H} W_n^{(2)} \tanh\left(W_n^{(1)}t_i + b_n^{(1)}\right) + b^{(2)}
$$

**First derivative of the neural network:**

$$
\frac{dNN}{dt}(t_i) =
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
$$

**Second derivative of the neural network:**

$$
\frac{d^2NN}{dt^2}(t_i) =
-2\sum_{n=1}^{H}
W_n^{(2)}(W_n^{(1)})^2
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
$$

**Loss function with the neural network and its derivatives fully substituted:**

$$
\begin{aligned}
L ={}&
\frac{1}{N_d}\sum_{i=1}^{N_d}
\Bigg(
\sum_{n=1}^{H}
W_n^{(2)}
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
+b^{(2)}
-e^{-2.5t_i}
\left(2\cdot0.5\cdot\cos(-0.11+22.22t_i)\right)
\Bigg)^2
\\[10pt]
&+\lambda_{phys}\frac{1}{N_c}
\sum_{i=1}^{N_c}
\Bigg[
-2\sum_{n=1}^{H}
W_n^{(2)}(W_n^{(1)})^2
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
\\[6pt]
&\qquad
+\frac{\mu}{m}
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(W_n^{(1)}t_i+b_n^{(1)}\right)
\right)
\\[6pt]
&\qquad
+\frac{k}{m}
\left(
\sum_{n=1}^{H}
W_n^{(2)}
\tanh\left(W_n^{(1)}t_i+b_n^{(1)}\right)
+b^{(2)}
\right)
\Bigg]^2
\\[10pt]
&+\lambda_{IC}
\Bigg[
\left(
\sum_{n=1}^{H}
W_n^{(2)}\tanh\left(b_n^{(1)}\right)
+b^{(2)}-1
\right)^2
\\[6pt]
&\qquad
+\left(
\sum_{n=1}^{H}
W_n^{(2)}W_n^{(1)}
\left(
1-\tanh^2\left(b_n^{(1)}\right)
\right)
\right)^2
\Bigg]
\end{aligned}
$$

Firstly, creating the neural network

In [3]:
#constants
m = 1.0 # mass
mu = 5 # friction constant
k = 500 # spring constant

omega_intial = np.sqrt(k)
delta = mu/(2*m)
omega = np.sqrt(omega_intial**2 - delta**2)

def A(phi):
    return 1 / (2*math.cos(phi))

phi = np.arctan(-1*delta/omega)

t_train = torch.linspace(start = 0, end = 0.5, steps = 10)
t_train = t_train.view(-1,1) # We transform it into shape (samples, features), instead of (samples). In our case that is (10,1)

def x(t):
    return  np.exp(-1*delta*t) *(2*A(phi)* np.cos(omega*t + phi))
y_train = x(t_train)

C:\Users\Saads\AppData\Local\Temp\ipykernel_44428\4019892306.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  return  np.exp(-1*delta*t) *(2*A(phi)* np.cos(omega*t + phi))


In [4]:
t_train = t_train.reshape(-1, 1)
t_train = t_train.detach().clone().requires_grad_(True).double()
t0 = torch.tensor([0.0], requires_grad=True)
# Creating the model
model = nn.Sequential(nn.Linear(1, 10), nn.Tanh(), nn.Linear(10, 1))
model = model.double()

# Making NN a function of t and then deriving it
# def NN(t):
#     return model(t)
# NN = model(t_train)
# dNN_dt = torch.autograd.grad(NN, t_train, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
# dNN2_dt2 = torch.autograd.grad(dNN_dt, t_train, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

# # Physics loss. The residuals must equal 0
# rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
# loss_phys = torch.mean(rt**2)

# # Initial conditions loss. x(0) = 1 and x'(0) = 0
# NN_0 = model(t0)
# dNN_0 = torch.autograd.grad(NN_0, t0, create_graph=True)[0]
# loss_IC1 = torch.mean((NN_0 - 1)**2)
# loss_IC2 = torch.mean(dNN_0**2)

# # Data loss. The model must provide the same output as the closed form solution
# # Version 1
# # loss_data = []
# # for i in range(len(t_train)):
# #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
# #     loss_data.append(data_loss_increment)
# # loss_data = torch.stack(loss_data).mean()
# # Version 2
# predictions = model(t_train)
# loss_fn = nn.MSELoss()
# assert predictions.shape == y_train.shape
# loss_data = loss_fn(predictions, y_train)


In [5]:
def calc_loss():
    t_col = torch.linspace(start = 0, end = 1, steps = 50).reshape(-1, 1).detach().clone().requires_grad_(True).double()
    NN = model(t_col)
    dNN_dt = torch.autograd.grad(NN, t_col, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
    dNN2_dt2 = torch.autograd.grad(dNN_dt, t_col, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

    # Physics loss. The residuals must equal 0
    rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
    loss_phys = torch.mean(rt**2)

    # Initial conditions loss. x(0) = 1 and x'(0) = 0
    t0 = torch.tensor([0.0], requires_grad=True).double()
    NN_0 = model(t0)
    dNN_0 = torch.autograd.grad(NN_0, t0, create_graph=True)[0]
    loss_IC1 = torch.mean((NN_0 - 1)**2)
    loss_IC2 = torch.mean(dNN_0**2)

    # Data loss. The model must provide the same output as the closed form solution
    # Version 1
    # loss_data = []
    # for i in range(len(t_train)):
    #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
    #     loss_data.append(data_loss_increment)
    # loss_data = torch.stack(loss_data).mean()
    # Version 2
    predictions = model(t_train)
    loss_fn = nn.MSELoss()
    
    assert predictions.shape == y_train.shape
    loss_data = loss_fn(predictions, y_train)
    
    lam_phys = 1e-4
    lam_ic = 1

    total_loss = loss_data + loss_phys * lam_phys + (loss_IC1 + loss_IC2) * lam_ic

    return total_loss


In [6]:
# print("Total loss shape: ", loss.shape)
# print("Total loss: ", loss.item())
# for i in range(10):
#     print(f'w({i}):', model[0].weight[i,0].item())
# Finite difference approximation

# w1_plus = model[0].weight.tolist()
# w1_minus = model[0].weight.tolist()
# b1 = model[0].bias.tolist()
# w2 = model[2].weight[0].tolist()
# b2 = model[2].bias.item()
e = 1e-5
fda_list_w1 = []
fda_list_w2 = []
l_plus_w1 = []
l_minus_w1 = []
l_plus_w2 = []
l_minus_w2 = []
loss_list_plus = []
loss_list_minus = []

for i in range(10):
    og_weight = model[0].weight[i,0].item()
    with torch.no_grad():
        model[0].weight[i, 0] = og_weight + e
    l_plus_w1.append(calc_loss().item())

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight - e
    l_minus_w1.append(calc_loss().item())

    with torch.no_grad():
        model[0].weight[i, 0] = og_weight

    fda_list_w1.append((l_plus_w1[-1] - l_minus_w1[-1]) / (2 * e))

for i in range(10):
    og_weight = model[2].weight[0, i].item()
    with torch.no_grad():
        model[2].weight[0, i] = og_weight + e
    l_plus_w2.append(calc_loss().item())

    with torch.no_grad():
        model[2].weight[0, i] = og_weight

    with torch.no_grad():
        model[2].weight[0, i] = og_weight - e
    l_minus_w2.append(calc_loss().item())

    with torch.no_grad():
        model[2].weight[0, i] = og_weight

    fda_list_w2.append((l_plus_w2[-1] - l_minus_w2[-1]) / (2 * e))

print("W1 Plus losses:", l_plus_w1)
print("W1 Minus losses:", l_minus_w1)
print("W2 Plus losses:", l_plus_w2)
print("W2 Minus losses:", l_minus_w2)
print("W1 Numerical gradient: ", fda_list_w1)
print("W2 Numerical gradient: ", fda_list_w2)

W1 Plus losses: [12.263657329093673, 12.26360845173578, 12.263626696228009, 12.263614416647902, 12.263670366644291, 12.263670104849936, 12.263617416655636, 12.263620481497965, 12.263635735874466, 12.26367226321126]
W1 Minus losses: [12.2636071710208, 12.263656048201568, 12.263637803664524, 12.263650083040078, 12.26359413354705, 12.263594394866649, 12.26364708306865, 12.263644018234666, 12.263628763936213, 12.26359223668216]
W2 Plus losses: [12.263387045089521, 12.263730147683725, 12.263855695326534, 12.263364288907175, 12.263441143019449, 12.263795083536069, 12.263372032564355, 12.263348405394728, 12.263919954245821, 12.263709219854524]
W2 Minus losses: [12.263877457451443, 12.263534352762306, 12.26340880682991, 12.263900214156028, 12.263823358612562, 12.263469417659854, 12.263892470345061, 12.263916098069176, 12.263344549308554, 12.263555280570097]
W1 Numerical gradient:  [2.5079036436892466, -2.3798232893668114, -0.5553718257722551, -1.7833196087835288, 3.811654862051483, 3.785499164

In [7]:
model.zero_grad()

total_loss = calc_loss()
total_loss.backward()

dW1 = model[0].weight.grad[:, 0].tolist()  # 10 first-layer gradients
dW2 = model[2].weight.grad[0, :].tolist()  # 10 second-layer gradients
print("dw1: ", dW1)
print("dw2: ", dW2)

dw1:  [2.5079036436096565, -2.3798232893782467, -0.555371825774364, -1.7833196087724341, 3.8116548621224875, 3.7854991643236473, -1.483320650632022, -1.17683683505616, 0.34859691291190487, 4.001326455063231]
dw2:  [-24.5206180959448, 9.789746070904723, 22.34442483134847, -26.796262442508585, -19.1107796557159, 16.28329381059608, -26.02188903540895, -28.38463372222038, 28.770246863369216, 7.696964221177001]


In [8]:
dW1 = np.asarray(dW1)
dW2 = np.asarray(dW2)
fda_list_w1 = np.asarray(fda_list_w1)
fda_list_w2 = np.asarray(fda_list_w2)

absolute_error_w1 = np.abs(dW1 - fda_list_w1)
absolute_error_w2 = np.abs(dW2 - fda_list_w2)
fig = px.line(y = [absolute_error_w1, absolute_error_w2], title='Absolute gradient error')
fig.update_layout(xaxis_title='Weight index', yaxis_title='Absolute error', template='plotly_dark', 
                 showlegend=True)
fig.show()

# 2.5

In [9]:
model_copy1 = copy.deepcopy(model)
model_copy2 = copy.deepcopy(model)

In [10]:

# data points and collocation points
t_data = np.linspace(0, 0.5, 10)
x_data = x(t_data)
t_col = np.linspace(0, 1, 50)


lam_ic = 1

def NN_manual(t, W1, b1, W2, b2):
    return np.sum(W2 * np.tanh(W1*t + b1)) + b2

def dNN(t, W1, b1, W2, b2):
    return np.sum(W2 * W1 * (1 - np.tanh(W1*t + b1)**2))

def ddNN(t, W1, b1, W2, b2):
    tanh = np.tanh(W1*t + b1)
    return np.sum(-2 * W2 * W1**2 * tanh * (1 - tanh**2))

def loss(W1, b1, W2, b2, lam_phys):
  # data term
  L_data = 0
  for i in range(len(t_data)):
      L_data += (NN_manual(t_data[i], W1, b1, W2, b2) - x_data[i])**2
  L_data = L_data / len(t_data)

  # physics term
  L_phys = 0
  for i in range(len(t_col)):
      t = t_col[i]
      r = ddNN(t, W1, b1, W2, b2) + mu/m * dNN(t, W1, b1, W2, b2) + k/m * NN_manual(t, W1, b1, W2, b2)
      L_phys += r**2
  L_phys = L_phys / len(t_col)

  # initial conditions
  L_ic = (NN_manual(0, W1, b1, W2, b2) - 1)**2 + (dNN(0, W1, b1, W2, b2) - 0)**2

  return L_data + lam_phys*L_phys + lam_ic*L_ic

def gradients(W1, b1, W2, b2, lam_phys):
    dW1 = np.zeros(len(W1))
    db1 = np.zeros(len(b1))
    dW2 = np.zeros(len(W2))
    db2 = 0

    # data term
    for i in range(len(t_data)):
        t = t_data[i]
        tanh = np.tanh(W1*t + b1)
        e = NN_manual(t, W1, b1, W2, b2) - x_data[i]

        dW1 += 2/len(t_data) * e * W2 * (1 - tanh**2) * t
        db1 += 2/len(t_data) * e * W2 * (1 - tanh**2)
        dW2 += 2/len(t_data) * e * tanh
        db2 += 2/len(t_data) * e

    # physics term
    for i in range(len(t_col)):
        t = t_col[i]
        tanh = np.tanh(W1*t + b1)
        r = ddNN(t, W1, b1, W2, b2) + mu/m * dNN(t, W1, b1, W2, b2) + k/m * NN_manual(t, W1, b1, W2, b2)

        # eq (42)
        dr_dW1 = (-4 * W2 * W1 * tanh * (1 - tanh**2)
                  - 2 * W2 * W1**2 * t * (1 - tanh**2)**2
                  + 4 * W2 * W1**2 * t * tanh**2 * (1 - tanh**2)
                  + mu/m * (W2 * (1 - tanh**2) - 2 * W2 * W1 * t * tanh * (1 - tanh**2))
                  + k/m * W2 * t * (1 - tanh**2))

        # eq (43)
        dr_dW2 = (-2 * W1**2 * tanh * (1 - tanh**2)
                  + mu/m * W1 * (1 - tanh**2)
                  + k/m * tanh)

        # biases: same as for W1 but without the factor t from the inner derivative
        dr_db1 = (-2 * W2 * W1**2 * (1 - tanh**2)**2
                  + 4 * W2 * W1**2 * tanh**2 * (1 - tanh**2)
                  + mu/m * (-2 * W2 * W1 * tanh * (1 - tanh**2))
                  + k/m * W2 * (1 - tanh**2))
        dr_db2 = k/m

        dW1 += lam_phys * 2/len(t_col) * r * dr_dW1
        db1 += lam_phys * 2/len(t_col) * r * dr_db1
        dW2 += lam_phys * 2/len(t_col) * r * dr_dW2
        db2 += lam_phys * 2/len(t_col) * r * dr_db2

    # initial conditions (t = 0)
    tanh0 = np.tanh(b1)
    NN0 = NN_manual(0, W1, b1, W2, b2)
    dNN0 = dNN(0, W1, b1, W2, b2)

    dW1 += lam_ic * 2 * dNN0 * W2 * (1 - tanh0**2)
    db1 += lam_ic * (2 * (NN0 - 1) * W2 * (1 - tanh0**2) - 4 * dNN0 * W2 * W1 * tanh0 * (1 - tanh0**2))
    dW2 += lam_ic * (2 * (NN0 - 1) * tanh0 + 2 * dNN0 * W1 * (1 - tanh0**2))
    db2 += lam_ic * 2 * (NN0 - 1)

    return dW1, db1, dW2, db2

In [11]:
def calc_loss(lam_phys):
    t_col = torch.linspace(start = 0, end = 1, steps = 50).reshape(-1, 1).detach().clone().requires_grad_(True).double()
    NN = model_copy2(t_col)
    dNN_dt = torch.autograd.grad(NN, t_col, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
    dNN2_dt2 = torch.autograd.grad(dNN_dt, t_col, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

    # Physics loss. The residuals must equal 0
    rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
    loss_phys = torch.mean(rt**2)

    # Initial conditions loss. x(0) = 1 and x'(0) = 0
    t0 = torch.tensor([0.0], requires_grad=True).double()
    NN_0 = model_copy2(t0)
    dNN_0 = torch.autograd.grad(NN_0, t0, create_graph=True)[0]
    loss_IC1 = torch.mean((NN_0 - 1)**2)
    loss_IC2 = torch.mean(dNN_0**2)

    # Data loss. The model must provide the same output as the closed form solution
    # Version 1
    # loss_data = []
    # for i in range(len(t_train)):
    #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
    #     loss_data.append(data_loss_increment)
    # loss_data = torch.stack(loss_data).mean()
    # Version 2
    predictions = model_copy2(t_train)
    loss_fn = nn.MSELoss()
    
    assert predictions.shape == y_train.shape
    loss_data = loss_fn(predictions, y_train)
    
    lam_ic = 1

    total_loss = loss_data + loss_phys * lam_phys + (loss_IC1 + loss_IC2) * lam_ic

    return total_loss


In [12]:
W1 = model_copy1[0].weight.detach().numpy()[:, 0]
b1 = model_copy1[0].bias.detach().numpy()
W2 = model_copy1[2].weight.detach().numpy()[0, :]
b2 = model_copy1[2].bias.detach().numpy()[0]

dW1, db1, dW2, db2 = gradients(W1, b1, W2, b2, lam_phys=1e-3)
manual_grads = np.concatenate([
    dW1.flatten(),
    db1.flatten(),
    dW2.flatten(),
    np.atleast_1d(db2)
])

model.zero_grad()
autograd_loss = calc_loss(lam_phys=1e-3)
autograd_tensors = torch.autograd.grad(autograd_loss, model_copy2.parameters())
autograd_grads = np.concatenate([gradient.detach().numpy().flatten() for gradient in autograd_tensors])

relative_error = np.linalg.norm(manual_grads - autograd_grads) / np.linalg.norm(manual_grads)
print(relative_error)

9.809136186077821e-10


In [13]:
model_copy1 = copy.deepcopy(model)
model_copy2 = copy.deepcopy(model)
model_copy3 = copy.deepcopy(model)

In [14]:
epoch = 20000
optimizer = torch.optim.Adam(model_copy1.parameters(), lr = 0.01)
losses = []

for i in range(epoch):
  # Loss function  where the forward pass also happens
  W1 = model_copy1[0].weight.detach().numpy()[:, 0]
  b1 = model_copy1[0].bias.detach().numpy()
  W2 = model_copy1[2].weight.detach().numpy()[0, :]
  b2 = model_copy1[2].bias.detach().numpy()[0]

  losses.append(loss(W1, b1, W2, b2, lam_phys=1e-3))

  # Update weights
  dW1, db1, dW2, db2 = gradients(W1, b1, W2, b2, lam_phys=1e-3)
  model_copy1[0].weight.grad = torch.tensor(dW1).reshape(10, 1)
  model_copy1[0].bias.grad = torch.tensor(db1)
  model_copy1[2].weight.grad = torch.tensor(dW2).reshape(1, 10)
  model_copy1[2].bias.grad = torch.tensor([db2], dtype=torch.float64)

  optimizer.step()

  if i % 1000 == 0:
      print(i, losses[-1])

optimizer = torch.optim.LBFGS(model_copy1.parameters(), lr=1.0, max_iter=1, line_search_fn="strong_wolfe")

def closure():
  optimizer.zero_grad()
  W1 = model_copy1[0].weight.detach().numpy()[:, 0]
  b1 = model_copy1[0].bias.detach().numpy()
  W2 = model_copy1[2].weight.detach().numpy()[0, :]
  b2 = model_copy1[2].bias.detach().numpy()[0]
  grad_loss = torch.tensor(loss(W1, b1, W2, b2, lam_phys=1e-3), dtype=torch.float64)
  dW1, db1, dW2, db2 = gradients(W1, b1, W2, b2, lam_phys=1e-3)
  model_copy1[0].weight.grad = torch.tensor(dW1).reshape(10, 1)
  model_copy1[0].bias.grad = torch.tensor(db1)
  model_copy1[2].weight.grad = torch.tensor(dW2).reshape(1, 10)
  model_copy1[2].bias.grad = torch.tensor([db2], dtype=torch.float64)
  return grad_loss

for i in range(500):
  losses.append(optimizer.step(closure).item())


0 115.26328349776287
1000 1.1986976125101878
2000 1.1970409688074382
3000 1.197023501099028
4000 1.196997109507566
5000 1.196954923941201
6000 1.1968610742105181
7000 1.1961340472875315
8000 1.1805555373845298
9000 1.0577093186551858
10000 0.8326752145751679
11000 0.8062994367673909
12000 0.7608860008907152
13000 0.6643179792495708
14000 0.501644000760943
15000 0.4268354140507688
16000 0.39631222919795456
17000 0.3706727101629168
18000 0.3482957248725983
19000 0.32724169826681354


## Changing regularization

In [15]:
epoch = 20000
lambda_values = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
models = [copy.deepcopy(model) for k in lambda_values]
optimizers = [torch.optim.Adam(m.parameters(), lr=0.01) for m in models]
loss_histories = [[] for k in lambda_values]

for i in range(epoch):
  for j, lam in enumerate(lambda_values):
    current_model = models[j]
    current_optimzer = optimizers[j]
    W1 = current_model[0].weight.detach().numpy()[:, 0]
    b1 = current_model[0].bias.detach().numpy()
    W2 = current_model[2].weight.detach().numpy()[0, :]
    b2 = current_model[2].bias.detach().numpy()[0]
  
    loss_histories[j].append(loss(W1, b1, W2, b2, lam))
  
    # Update weights
    dW1, db1, dW2, db2 = gradients(W1, b1, W2, b2, lam)
    current_model[0].weight.grad = torch.tensor(dW1).reshape(10, 1)
    current_model[0].bias.grad = torch.tensor(db1)
    current_model[2].weight.grad = torch.tensor(dW2).reshape(1, 10)
    current_model[2].bias.grad = torch.tensor([db2], dtype=torch.float64)
  
    current_optimzer.step()

for j, lam in enumerate(lambda_values):
  current_model = models[j]
  current_optimzer = torch.optim.LBFGS(current_model.parameters(), lr=1.0, max_iter=1, line_search_fn="strong_wolfe")

  def closure():
    current_optimzer.zero_grad()
    W1 = current_model[0].weight.detach().numpy()[:, 0]
    b1 = current_model[0].bias.detach().numpy()
    W2 = current_model[2].weight.detach().numpy()[0, :]
    b2 = current_model[2].bias.detach().numpy()[0]
    grad_loss = torch.tensor(loss(W1, b1, W2, b2, lam), dtype=torch.float64)
    dW1, db1, dW2, db2 = gradients(W1, b1, W2, b2, lam)
    current_model[0].weight.grad = torch.tensor(dW1).reshape(10, 1)
    current_model[0].bias.grad = torch.tensor(db1)
    current_model[2].weight.grad = torch.tensor(dW2).reshape(1, 10)
    current_model[2].bias.grad = torch.tensor([db2], dtype=torch.float64)
    return grad_loss

  for i in range(500):
    loss_histories[j].append(current_optimzer.step(closure).item())

fig = go.Figure()
for lam, reg_loss in zip(lambda_values, loss_histories):
  fig.add_scatter(x=list(range(len(reg_loss))), y=reg_loss, mode='lines', name=f'lam={lam:g}')
fig.update_layout(title='Different regularizations', xaxis_title="Epoch", yaxis_title="Losses", template="plotly_dark")
fig.show()

In [16]:
t_eval = torch.linspace(0, 1, 100).reshape(-1,1).double()
reg_predictions = [[] for k in lambda_values]
with torch.no_grad():
    for i in range(len(lambda_values)):
        reg_predictions[i] = models[i](t_eval).numpy().flatten()

fig = go.Figure()
for lam, reg_pred in zip(lambda_values, reg_predictions):
  fig.add_scatter(x=t_eval.flatten(), y=reg_pred, mode='lines', name=f'lam={lam:g}')
fig.update_layout(title='Inference of different regularization', xaxis_title="Time", yaxis_title="Diplacement", template="plotly_dark")
fig.show()

# Task 2.6

In [17]:
loss_list = []
optimizer = torch.optim.Adam(model_copy2.parameters(), lr = 0.01)
for i in range(epoch):
    model_copy2.zero_grad()
    grad_loss = (calc_loss(lam_phys=1e-3))
    loss_list.append(grad_loss.item())
    grads = torch.autograd.grad(grad_loss, model_copy2.parameters())
    for parameter, gradient in zip(model_copy2.parameters(), grads):
        parameter.grad = gradient
    optimizer.step()
    if i % 1000 == 0:
        print(i, loss_list[-1])

optimizer = torch.optim.LBFGS(model_copy2.parameters(), lr=1.0, max_iter=1, line_search_fn="strong_wolfe")

def closure():
    optimizer.zero_grad()
    grad_loss = calc_loss(lam_phys=1e-3)
    grads = torch.autograd.grad(grad_loss, model_copy2.parameters())
    for parameter, gradient in zip(model_copy2.parameters(), grads):
        parameter.grad = gradient
    return grad_loss

for i in range(500):
    loss_list.append(optimizer.step(closure).item())


0 115.26328334435324
1000 1.1986975923390721
2000 1.1970409487021691
3000 1.1970234809945528
4000 1.1969970894041184
5000 1.1969549038392135
6000 1.1968610541136284
7000 1.1961336632331225
8000 1.1806065832377
9000 1.0618133018093014
10000 0.832633417231692
11000 0.8023193044827779
12000 0.7599388620675154
13000 0.6606540977838595
14000 0.5012674487854986
15000 0.5740143829732215
16000 0.3955279846553286
17000 0.37017797390613755
18000 0.34773992695177525
19000 0.32693343739587943


In [18]:
loss_array = np.asarray(loss_list)
losses_array = np.asarray(losses)

In [19]:
abs_differnce_losses = np.abs(losses_array - loss_array)

fig = px.line(y = abs_differnce_losses, title='Absolute error losses')
fig.update_layout(xaxis_title='Index', yaxis_title='Absolute error', template='plotly_dark', 
                 showlegend=True)
fig.show()

In [20]:
fig = px.line(y=[losses_array, loss_array], title="Losses comparison")
fig.update_layout(xaxis_title='Index', yaxis_title='Lossesr', template='plotly_dark', 
                 showlegend=True)
fig.show()

In [21]:
t_eval = torch.linspace(0, 1, 100).reshape(-1,1).double()
with torch.no_grad():
    manual_predictions = model_copy1(t_eval).numpy().flatten()
    autograd_predictions = model_copy2(t_eval).numpy().flatten()
fig = px.line(x=t_eval.flatten(), y=[manual_predictions, autograd_predictions], title='Inference comparison')
fig.update_layout(xaxis_title='Time', yaxis_title='Displacement', template='plotly_dark', 
                 showlegend=True)
fig.show()

# Task 2.7

In [35]:
model_hard = copy.deepcopy(model)
def x_0(t):
    return 1 + t**2 * model_hard(t)

In [36]:
t = torch.tensor([0.0], requires_grad=True, dtype=torch.float64)
value = x_0(t).backward()
print(x_0(t).item())
print(t.grad.item())

1.0
0.0


In [37]:

def calc_loss_hard(lam_phys):
    t_col = torch.linspace(start = 0, end = 1, steps = 50).reshape(-1, 1).detach().clone().requires_grad_(True).double()
    NN = x_0(t_col)
    dNN_dt = torch.autograd.grad(NN, t_col, grad_outputs=torch.ones_like(NN), create_graph=True)[0]
    dNN2_dt2 = torch.autograd.grad(dNN_dt, t_col, grad_outputs=torch.ones_like(dNN_dt), create_graph=True)[0]

    # Physics loss. The residuals must equal 0
    rt = dNN2_dt2 + (mu/m)*dNN_dt + (k/m)*NN
    loss_phys = torch.mean(rt**2)

    # Initial conditions loss. x(0) = 1 and x'(0) = 0


    # Data loss. The model must provide the same output as the closed form solution
    # Version 1
    # loss_data = []
    # for i in range(len(t_train)):
    #     data_loss_increment = torch.mean((model(t_train[i]) - y_train[i])**2)
    #     loss_data.append(data_loss_increment)
    # loss_data = torch.stack(loss_data).mean()
    # Version 2
    predictions = x_0(t_train)
    loss_fn = nn.MSELoss()
    
    assert predictions.shape == y_train.shape
    loss_data = loss_fn(predictions, y_train)


    total_loss = loss_data + lam_phys * loss_phys

    return total_loss


In [38]:
loss_list_hard = []
optimizer = torch.optim.Adam(model_hard.parameters(), lr = 0.01)
for i in range(epoch):
    model_hard.zero_grad()
    grad_loss_hard = (calc_loss_hard(lam_phys=1e-3))
    loss_list_hard.append(grad_loss_hard.item())
    grads = torch.autograd.grad(grad_loss_hard, model_hard.parameters())
    for parameter, gradient in zip(model_hard.parameters(), grads):
        parameter.grad = gradient
    optimizer.step()
    if i % 1000 == 0:
        print(i, loss_list_hard[-1])

optimizer = torch.optim.LBFGS(model_hard.parameters(), lr=1.0, max_iter=1, line_search_fn="strong_wolfe")

def closure():
    optimizer.zero_grad()
    grad_loss_hard = calc_loss_hard(lam_phys=1e-3)
    grads = torch.autograd.grad(grad_loss_hard, model_hard.parameters())
    for parameter, gradient in zip(model_hard.parameters(), grads):
        parameter.grad = gradient
    return grad_loss_hard

for i in range(500):
    loss_list_hard.append(optimizer.step(closure).item())


0 399.65061600652245
1000 35.440851148152966
2000 23.840749897949188
3000 18.056589445384752
4000 13.977309684367402
5000 11.13923650619229
6000 9.161191516580132
7000 7.786318147450972
8000 6.8950709658059495
9000 6.266504037173438
10000 5.785429874194833
11000 5.400018489638908
12000 5.079610524516767
13000 4.805124164028761
14000 4.5654445844424725
15000 4.352988254968636
16000 4.163364028992898
17000 3.992619016859285
18000 3.8398498424540586
19000 3.7045620156950934


In [44]:
t_eval_hard = torch.linspace(0.5, 1, 100, dtype=torch.float64).reshape(-1,1)
with torch.no_grad():
    with torch.no_grad():
        hard_predictions = x_0(t_eval_hard).flatten()
        soft_pred = model_copy2(t_eval_hard).flatten()
        analytical_predictions = x(t_eval_hard).flatten()
    fig = go.Figure()
    fig.add_scatter(x=t_eval_hard.flatten(), y=hard_predictions, mode="lines", name="Hard enforcement")
    fig.add_scatter(x=t_eval_hard.flatten(), y=analytical_predictions, mode="lines", name="Analytical")
    fig.add_scatter(x=t_eval_hard.flatten(), y=soft_pred, mode='lines', name='Soft Initial conditions')
    fig.update_layout(
        title="Inference",
        xaxis_title="Time",
        yaxis_title="Displacement",
        template="plotly_dark"
    )
fig.show()

C:\Users\Saads\AppData\Local\Temp\ipykernel_44428\4019892306.py:19: DeprecationWarning:

__array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)



In [27]:
fig = go.Figure()
fig.add_scatter(y=loss_list, mode='lines', name='Soft Initial conditions')
fig.add_scatter(y=loss_list_hard, mode='lines', name='Hard Initial conditions')
fig.update_layout(
    title="Losses",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    template="plotly_dark"
)
fig.show()

In [28]:
mse_hard = torch.mean((hard_predictions - analytical_predictions)**2)
mse_soft = torch.mean((soft_pred - analytical_predictions)**2)

print("Hard IC MSE:", mse_hard.item())
print("Soft IC MSE:", mse_soft.item())

Hard IC MSE: 0.04337366345763142
Soft IC MSE: 0.02070815081716488


the soft initial-condition PINN achieved better overall prediction accuracy than the hard-constraint PINN in this experimen